In [ ]:
import sys
import subprocess
import importlib
import threading
import socket
import time
import struct
import os
import csv
import json
import shutil
import math
from datetime import datetime

# ---------------------------------------------------------
# 1. 패키지 자동 설치 함수
# ---------------------------------------------------------
def install_package(module_name, package_name=None):
    if package_name is None:
        package_name = module_name
    try:
        importlib.import_module(module_name)
    except ImportError:
        print(f"Installing {package_name} ...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])
            print(f"{package_name} installation completed")
        except subprocess.CalledProcessError as e:
            print(f"{package_name} installation failed (exit code {e.returncode})")
            sys.exit(1)

print("Checking required packages...")
install_package("numpy")
install_package("cv2", "opencv-python")
install_package("PyQt5")

import numpy as np
import cv2
from PyQt5.QtWidgets import *
from PyQt5.QtGui import *
from PyQt5.QtCore import *
from PyQt5 import QtWidgets, QtGui

# ---------------------------------------------------------
# 통합된 조이스틱 클래스
# ---------------------------------------------------------
class MyJoystick(QWidget):
    def __init__(self, cbJoyPos=None, parent=None):
        super(MyJoystick, self).__init__(parent)
        self.setMinimumSize(200, 200)
        self.movingOffset = QPointF(0, 0)
        self.grabCenter = False
        self.__maxDistance = 50

        self.timer = QTimer(self)
        self.timer.setInterval(10)
        self.timer.timeout.connect(self.timeout)
        self.timer.start()
        
        self.cbJoyPos = cbJoyPos

    def paintEvent(self, event):
        painter = QPainter(self)
        bounds = QRectF(
        -self.__maxDistance, 
        -self.__maxDistance, 
        self.__maxDistance * 2, 
        self.__maxDistance * 2
        ).translated(self._center())
        painter.drawEllipse(bounds)
        painter.setBrush(Qt.black)
        painter.drawEllipse(self._centerEllipse())

    def _centerEllipse(self):
        if self.grabCenter:
            return QRectF(-20, -20, 40, 40).\
            translated(self.movingOffset)
        return QRectF(-20, -20, 40, 40).translated(self._center())

    def _center(self):
        return QPointF(self.width()/2, self.height()/2)

    def _boundJoystick(self, point):
        limitLine = QLineF(self._center(), point)
        if (limitLine.length() > self.__maxDistance):
            limitLine.setLength(self.__maxDistance)
        return limitLine.p2()

    def joystickPosition(self):
        if not self.grabCenter:
            return (0, 0)
        normVector = QLineF(self._center(), self.movingOffset)
        currentDistance = normVector.length()
        angle = normVector.angle()

        distance = min(currentDistance / self.__maxDistance, 1.0)

        posX = math.cos(angle*math.pi/180)*distance
        posY = math.sin(angle*math.pi/180)*distance

        return (posX, posY)

    def mousePressEvent(self, ev):
        self.grabCenter = self._centerEllipse().contains(ev.pos())
        return super().mousePressEvent(ev)

    def mouseReleaseEvent(self, event):
        self.grabCenter = False
        self.movingOffset = QPointF(0, 0)
        self.update()

    def mouseMoveEvent(self, event):
        if self.grabCenter:
            self.movingOffset = self._boundJoystick(event.pos())
            self.update()
        if self.cbJoyPos != None :
            self.cbJoyPos(self.joystickPosition())

    def timeout(self):
        sender = self.sender()
        if id(sender) == id(self.timer):
            if self.cbJoyPos != None :
                self.cbJoyPos(self.joystickPosition())

# ---------------------------------------------------------
# 2. 윈도우 소켓 버그 수정용 함수
# ---------------------------------------------------------
def recvall(sock, count):
    buf = b''
    while len(buf) < count:
        try:
            newbuf = sock.recv(count - len(buf))
            if not newbuf: return None
            buf += newbuf
        except socket.timeout:
            raise
        except BlockingIOError:
            continue
    return buf

# ---------------------------------------------------------
# 전역 변수 설정
# ---------------------------------------------------------
HOST_CAM = '192.168.0.60' # ESP32 IP
PORT_CAM = 80
PORT_MOT = 81

client_cam = None
client_mot = None

running = False
label_widget = None
camera_thread = None

dirname = ""
f_csv = None
wr = None
labels_list = ["_0_forward", "_1_right", "_2_left", "_3_stop"]

g_current_label = 3   # 0:Fwd, 1:Right, 2:Left, 3:Stop
g_is_recording = False

CONFIG_FILE = "mask_config.json"
crop_top = 0
crop_bottom = 0
crop_left = 0
crop_right = 0
flip_h = True
flip_v = False

# ---------------------------------------------------------
# 설정 저장/로드 함수
# ---------------------------------------------------------
def save_mask_config():
    config = {
        "top": crop_top,
        "bottom": crop_bottom,
        "left": crop_left,
        "right": crop_right,
        "flip_h": flip_h,
        "flip_v": flip_v
    }
    try:
        with open(CONFIG_FILE, 'w') as f:
            json.dump(config, f)
        print(f"Mask config saved to {CONFIG_FILE}")
    except Exception as e:
        print(f"Failed to save config: {e}")

def load_mask_config():
    global crop_top, crop_bottom, crop_left, crop_right, flip_h, flip_v
    if os.path.exists(CONFIG_FILE):
        try:
            with open(CONFIG_FILE, 'r') as f:
                config = json.load(f)
                crop_top = config.get("top", 0)
                crop_bottom = config.get("bottom", 0)
                crop_left = config.get("left", 0)
                crop_right = config.get("right", 0)
                flip_h = config.get("flip_h", True)
                flip_v = config.get("flip_v", True)
            print(f"Loaded mask config: {config}")
            return True
        except Exception as e:
            print(f"Failed to load config: {e}")
    return False

# ---------------------------------------------------------
# 소켓 연결 함수
# ---------------------------------------------------------
def try_connect_esp32(ip_address):
    global client_cam, client_mot
    print(f"Connecting to ESP32 ({ip_address})...")
    
    if client_cam:
        try: client_cam.close()
        except: pass
    if client_mot:
        try: client_mot.close()
        except: pass

    try:
        cam_sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        mot_sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        mot_sock.setsockopt(socket.IPPROTO_TCP, socket.TCP_NODELAY, 1)
        
        cam_sock.settimeout(3)
        mot_sock.settimeout(3)

        cam_sock.connect((ip_address, PORT_CAM))
        mot_sock.connect((ip_address, PORT_MOT))
        
        cam_sock.settimeout(60) 
        mot_sock.settimeout(60)
        
        client_cam = cam_sock
        client_mot = mot_sock
        print("Connected successfully!")
        return True
    except Exception as e:
        print(f"Connection Failed: {e}")
        return False

# ---------------------------------------------------------
# 폴더 관리
# ---------------------------------------------------------
def createNewFolder():
    global dirname, f_csv, wr
    if f_csv:
        try: f_csv.close()
        except: pass

    dirname = datetime.now().strftime("%Y%m%d_%H_%M_%S")
    if not os.path.exists(dirname):
        os.mkdir(dirname)
    
    for label in labels_list:
        path = os.path.join(dirname, label)
        if not os.path.exists(path):
            os.mkdir(path)

    f_csv = open(os.path.join(dirname, "0_road_labels.csv"),'w', newline='')
    wr = csv.writer(f_csv)
    wr.writerow(["file","label"])
    
    save_mask_config()
    try: shutil.copy(CONFIG_FILE, os.path.join(dirname, CONFIG_FILE))
    except: pass

    print(f"--> New data folder created: {dirname}")

def continueFromFolder(path):
    global dirname, f_csv, wr
    if f_csv:
        try: f_csv.close()
        except: pass
    
    dirname = path
    print(f"--> Continuing in folder: {dirname}")

    for label in labels_list:
        subpath = os.path.join(dirname, label)
        if not os.path.exists(subpath):
            os.mkdir(subpath)

    csv_path = os.path.join(dirname, "0_road_labels.csv")
    file_exists = os.path.exists(csv_path)

    f_csv = open(csv_path, 'a', newline='')
    wr = csv.writer(f_csv)

    if not file_exists:
        wr.writerow(["file","label"])
        print("--> Created new CSV file in existing folder.")
    else:
        print("--> Appending to existing CSV file.")

def selectFolderMode():
    msg = QMessageBox()
    msg.setWindowTitle("Data Collection Mode")
    msg.setText("Choose where to save data:")
    msg.setIcon(QMessageBox.Question)
    
    btn_new = msg.addButton("Create New Folder", QMessageBox.AcceptRole)
    btn_cont = msg.addButton("Continue Existing Folder", QMessageBox.AcceptRole)
    
    msg.exec_()
    
    if msg.clickedButton() == btn_cont:
        folder = QFileDialog.getExistingDirectory(None, "Select Folder to Continue")
        if folder:
            continueFromFolder(folder)
        else:
            print("Selection cancelled. Creating new folder.")
            createNewFolder()
    else:
        createNewFolder()

# ---------------------------------------------------------
# 카메라 쓰레드
# ---------------------------------------------------------
def camMain():
    global running, g_current_label, g_is_recording, dirname, f_csv, wr
    global crop_top, crop_bottom, crop_left, crop_right, flip_h, flip_v
    
    t_prev = time.time()
    cnt_frame = 0
    cnt_frame_total = 0
    
    DISPLAY_WIDTH = 640
    DISPLAY_HEIGHT = 480
    
    while label_widget is None and running:
        time.sleep(0.1)
    
    try:
        if label_widget: label_widget.resize(DISPLAY_WIDTH, DISPLAY_HEIGHT)
    except: pass

    while running:
        if client_cam is None:
            time.sleep(1)
            continue

        try:
            cmd = 12
            cmd = struct.pack('B', cmd)
            client_cam.sendall(cmd) 

            data_len_bytes = recvall(client_cam, 4)
            if not data_len_bytes: continue
            data_len = struct.unpack('I', data_len_bytes)[0]
            
            data = recvall(client_cam, data_len)
            if not data: continue

            np_data = np.frombuffer(data, dtype='uint8')
            frame = cv2.imdecode(np_data, 1)
            if frame is None: continue

            if flip_h and flip_v:
                frame = cv2.flip(frame, -1)
            elif flip_h:
                frame = cv2.flip(frame, 1)
            elif flip_v:
                frame = cv2.flip(frame, 0)
                
            frame_resized = cv2.resize(frame, (DISPLAY_WIDTH, DISPLAY_HEIGHT), interpolation=cv2.INTER_LINEAR)
            
            h, w, _ = frame_resized.shape
            if crop_top > 0: frame_resized[:crop_top, :] = 0
            if crop_bottom > 0: frame_resized[h-crop_bottom:, :] = 0
            if crop_left > 0: frame_resized[:, :crop_left] = 0
            if crop_right > 0: frame_resized[:, w-crop_right:] = 0
            
            qImg = QtGui.QImage(frame_resized.data, w, h, w*3, QtGui.QImage.Format_RGB888)
            pixmap = QtGui.QPixmap.fromImage(qImg.rgbSwapped())
            
            if label_widget is not None:
                label_widget.setPixmap(pixmap)
            
            if g_is_recording and f_csv is not None and not f_csv.closed:
                label_idx = g_current_label
                time_str = datetime.now().strftime("%f") 
                road_file = f"{time.time()}.png"
                
                save_path = os.path.join(dirname, labels_list[label_idx])
                
                cv2.imwrite(os.path.join(save_path, road_file), frame_resized)
                
                if wr:
                    wr.writerow([os.path.join(labels_list[label_idx], road_file), label_idx])
            
                cnt_frame_total += 1

            cnt_frame += 1
            t_now = time.time()
            if t_now - t_prev >= 1.0 :
                status_str = "REC" if g_is_recording else "IDLE"
                print(f"FPS: {cnt_frame}, Total Saved: {cnt_frame_total}, State: {status_str}, Label: {labels_list[g_current_label]}")
                t_prev = t_now
                cnt_frame = 0
                
        except socket.timeout:
            print("Socket timed out.")
            time.sleep(0.5)
            continue
        except Exception as e:
            print(f"Cam Error: {e}")
            break

# ---------------------------------------------------------
# 조이스틱
# ---------------------------------------------------------
def cbJoyPos(joystickPosition):
    global g_current_label, g_is_recording
    
    if client_mot is None: return

    posX, posY = joystickPosition
    MIN_PWM = 250
    MAX_PWM = 1023

    def map_speed(value):
        val = abs(value)
        if val < 0.15: val = 0.15
        ratio = (val - 0.15) / (1.0 - 0.15)
        speed = int(MIN_PWM + (MAX_PWM - MIN_PWM) * ratio)
        if speed > MAX_PWM: speed = MAX_PWM
        if speed < MIN_PWM: speed = MIN_PWM
        return speed

    cmd_char = b'S\n'
    
    # 1. 전진 및 회전 (위로 밈 -> 녹화 O)
    if posY > 0.15:
        g_is_recording = True 
        
        if posX < -0.3: # 좌회전
            speed = map_speed(posX)
            cmd_char = f'L,{speed}\n'.encode()
            g_current_label = 2 
            
        elif posX > 0.3: # 우회전
            speed = map_speed(posX)
            cmd_char = f'R,{speed}\n'.encode()
            g_current_label = 1 
            
        else: # 직진
            speed = map_speed(posY)
            cmd_char = f'F,{speed}\n'.encode()
            g_current_label = 0 

    # 2. 후진 (아래로 당김 -> 녹화 X)
    elif posY < -0.15:
        g_is_recording = False 
        speed = map_speed(posY)
        cmd_char = f'B,{speed}\n'.encode()
        g_current_label = 3 

    # 3. 정지 (중앙 -> 녹화 X)
    else:
        g_is_recording = False
        cmd_char = b'S\n'
        g_current_label = 3 

    try:
        client_mot.sendall(cmd_char)
    except: pass

# ---------------------------------------------------------
# 메인 윈도우 클래스
# ---------------------------------------------------------
class MainWindow(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle('RC Car Joystick Data Collector (Masking Supported)')
        self.setGeometry(100, 100, 500, 700) 

        cw = QWidget()
        self.setCentralWidget(cw)
        layout = QGridLayout()
        cw.setLayout(layout)

        # 1. 화면 Label
        global label_widget
        label_widget = QLabel("Waiting for Video...")
        label_widget.setAlignment(Qt.AlignCenter)
        label_widget.setStyleSheet("background-color: black; color: white;")
        label_widget.setScaledContents(True)
        # 640x480 비율 유지
        label_widget.setMinimumSize(320, 240) 
        layout.addWidget(label_widget, 0, 0, 1, 2)

        gb_mask = QGroupBox("Camera Masking Settings")
        gb_layout = QGridLayout()
        gb_mask.setLayout(gb_layout)

        def create_slider(label_text, row, col, max_val, callback, init_val):
            lbl = QLabel(label_text)
            slider = QSlider(Qt.Horizontal)
            slider.setRange(0, max_val)
            slider.setValue(init_val)
            val_label = QLabel(str(init_val))
            
            def on_change(val):
                val_label.setText(str(val))
                callback(val)
                
            slider.valueChanged.connect(on_change)
            
            gb_layout.addWidget(lbl, row, col)
            gb_layout.addWidget(slider, row, col+1)
            gb_layout.addWidget(val_label, row, col+2)
            return slider

        create_slider("Top:", 0, 0, 240, self.set_crop_top, crop_top)
        create_slider("Bottom:", 1, 0, 240, self.set_crop_bottom, crop_bottom)
        create_slider("Left:", 0, 3, 320, self.set_crop_left, crop_left)
        create_slider("Right:", 1, 3, 320, self.set_crop_right, crop_right)

        layout.addWidget(gb_mask, 1, 0, 1, 2)

        flip_layout = QHBoxLayout()
        self.chk_flip_h = QCheckBox("좌우 반전 (Horizontal Flip)")
        self.chk_flip_v = QCheckBox("상하 반전 (Vertical Flip)")
        
        self.chk_flip_h.setChecked(flip_h)
        self.chk_flip_v.setChecked(flip_v)
        
        self.chk_flip_h.stateChanged.connect(self.on_flip_h_changed)
        self.chk_flip_v.stateChanged.connect(self.on_flip_v_changed)
        
        flip_layout.addWidget(self.chk_flip_h)
        flip_layout.addWidget(self.chk_flip_v)
        layout.addLayout(flip_layout, 2, 0, 1, 2)

        self.joystick = MyJoystick(cbJoyPos)
        layout.addWidget(self.joystick, 3, 0, 1, 2)

        self.btn_new_folder = QPushButton("Make New Folder (N)")
        self.btn_new_folder.setFixedHeight(40)
        self.btn_new_folder.clicked.connect(self.on_new_folder_clicked)
        layout.addWidget(self.btn_new_folder, 4, 0, 1, 2)
        
        self.btn_change_ip = QPushButton(f"Change IP (Current: {HOST_CAM})")
        self.btn_change_ip.setFixedHeight(40)
        self.btn_change_ip.clicked.connect(self.on_change_ip_clicked)
        layout.addWidget(self.btn_change_ip, 5, 0, 1, 2)

        info_label = QLabel("Moving joystick UP/LEFT/RIGHT records data.\nSettings are saved to mask_config.json automatically.")
        info_label.setAlignment(Qt.AlignCenter)
        layout.addWidget(info_label, 6, 0, 1, 2)

    def set_crop_top(self, val): global crop_top; crop_top = val
    def set_crop_bottom(self, val): global crop_bottom; crop_bottom = val
    def set_crop_left(self, val): global crop_left; crop_left = val
    def set_crop_right(self, val): global crop_right; crop_right = val

    def on_flip_h_changed(self, state):
        global flip_h
        flip_h = (state == Qt.Checked)

    def on_flip_v_changed(self, state):
        global flip_v
        flip_v = (state == Qt.Checked)

    def on_new_folder_clicked(self):
        createNewFolder()
        QMessageBox.information(self, "New Folder", f"New folder created:\n{dirname}")

    def on_change_ip_clicked(self):
        global HOST_CAM, running, camera_thread
        text, ok = QInputDialog.getText(self, 'Change IP', 'Enter ESP32 IP Address:', text=HOST_CAM)
        
        if ok and text:
            HOST_CAM = text
            self.btn_change_ip.setText(f"Change IP (Current: {HOST_CAM})")
            running = False
            if camera_thread: camera_thread.join(timeout=1.0)
            
            if try_connect_esp32(HOST_CAM):
                QMessageBox.information(self, "Success", "Connected to new IP!")
                running = True
                camera_thread = threading.Thread(target=camMain)
                camera_thread.setDaemon(True); camera_thread.start()
            else:
                QMessageBox.warning(self, "Failed", "Could not connect.")

    def keyPressEvent(self, event):
        if event.key() == Qt.Key_Escape: self.close()
        elif event.key() == Qt.Key_N: self.on_new_folder_clicked()

    def closeEvent(self, event):
        print("Window closing...")
        save_mask_config() 
        event.accept()

# ---------------------------------------------------------
# 실행부
# ---------------------------------------------------------
app = QApplication.instance()
if app is None:
    app = QApplication(sys.argv)
app.setStyle(QStyleFactory.create("Cleanlooks"))

load_mask_config()

connected = False
while not connected:
    connected = try_connect_esp32(HOST_CAM)
    if not connected:
        text, ok = QInputDialog.getText(None, 'Connection Failed', 'Enter ESP32 IP Address:', text=HOST_CAM)
        if ok and text: HOST_CAM = text 
        else: sys.exit()

if connected:
    selectFolderMode()
    
    mw = MainWindow()
    mw.show()

    running = True
    camera_thread = threading.Thread(target=camMain)
    camera_thread.setDaemon(True) 
    camera_thread.start()

    try:
        app.exec_()
    except SystemExit:
        pass
    finally:
        running = False
        try: client_cam.close() 
        except: pass
        try: client_mot.close() 
        except: pass
        if f_csv:
            try: f_csv.close() 
            except: pass
        print("Done.")